# 01. Decisions

**Tier:** Intermediate. **Why you'd use this:** You're labeling a traffic light for a driving drill. Red means stop. Yellow and amber both mean slow. Every other color means go. One value, a few shapes, and some shapes share a meaning.

**After this lesson you can:** give two color names the same result; send every leftover color to one result; catch a bare name that matches every color and overwrites your variable.
About 25 minutes. **Uses:** [Conditions](../01-basics/03-conditions-lesson.ipynb)


## Quick review

1. What does `0 or "go"` return?
2. Why is `"hello"[1:4]` three characters?

<details><summary>Answers</summary>

`or` returns the first operand that counts as true, so the string `"go"`, not the word True. The slice leaves out its stop.
</details>


## The idea

`match` takes one value and tries patterns from top to bottom. The first pattern that fits runs its arm. A quoted pattern fits only that text. `|` puts two patterns on one arm. `case _` fits whatever is left, and it stores nothing.

Think of sorting mail into labeled bins. The letter goes in the first bin whose label fits. `_` is the bin marked "everything else."

<details><summary>Explain more</summary>

Each `case` ends with a colon. The lines under it are the arm. A bare name, with no quotes, is not a label you already defined. It is a capture: it fits any value and stores that value under the name. If nothing fits and there is no `_`, the `match` ends and the function returns `None`.
</details>


## See it work

Label the light from the drill.


In [1]:
def light(color):
    # 1. The one color that means stop.
    match color:
        case "red":
            return "stop"
        # 2. Two names, one meaning.
        case "yellow" | "amber":
            return "slow"
        # 3. Every color that was not named.
        case _:
            return "go"


print(light("red"), light("amber"), light("blue"))


stop slow go


## Check yourself

Answer these before you run the cell.

1. Which arm handles `"amber"`, and why doesn't that color fall through to `"go"`?
2. What does `light("blue")` return?
3. In the cell, what is `ok` after the match, and why did `"missing"` count as fine?


In [2]:
print(light("amber"), light("blue"))
ok = "ok"
status = "missing"
match status:
    case ok:
        seen = "fine"
print(seen)
print(ok)


slow go
fine
missing


<details><summary>Answers</summary>

`"amber"` fits the second arm, and `match` stops there. `"blue"` fits `_`, so the result is `"go"`. The bare name `ok` did not look up the variable. It stored `"missing"` into `ok`, so the arm ran and the old string is gone.
</details>


## Why it works this way

`match` is structural pattern matching. It asks "does this value have this shape?", in order. It is not a C `switch`. A switch jumps from an integer into a table. These arms are tests. For quoted patterns, CPython compiles each one as an equality check against the subject, then goes to the next arm only if that check failed. The `dis` listing shows that sequence. Walking arms until one fits is the cost: a match near the bottom does every comparison above it. A subject that misses every specific arm still pays for those comparisons. Put the patterns you expect first.

A bare name is a capture pattern. It fits every value, binds that value to the name, and does not read the variable you had. That is why `case ok` treats `"missing"` as a hit and why `ok` changes. `_` also fits everything, but it binds nothing, so it can be the leftover bin. A pattern that already fits everything makes later arms dead. Compiling a wildcard and then another arm fails for that reason.

A quoted or numeric pattern uses `==`, not `is`. `True` equals `1`, so `case 1` matches `True`. The same subclass relationship from conditions shows up as a pattern. `|` tries its sides from left to right and stops at the first fit. It does not combine bits. The design is that order is priority, and a capture is a place to keep the value, not a way to mention an existing variable. Deep dive: [Source to bytecode](../../internals/10-bytecode/01-bytecode-lesson.ipynb).


In [3]:
import dis

dis.dis(light)


  1           RESUME                   0

  3           LOAD_FAST                0 (color)

  4           COPY                     1
              LOAD_CONST               0 ('red')
              COMPARE_OP              88 (bool(==))
              POP_JUMP_IF_FALSE        4 (to L1)
              NOT_TAKEN

  5           POP_TOP
              LOAD_CONST               1 ('stop')
              RETURN_VALUE

  7   L1:     COPY                     1
              LOAD_CONST               2 ('yellow')
              COMPARE_OP              88 (bool(==))
              POP_JUMP_IF_FALSE        4 (to L2)
              NOT_TAKEN
              POP_TOP

  8           LOAD_CONST               4 ('slow')
              RETURN_VALUE

  7   L2:     COPY                     1
              LOAD_CONST               3 ('amber')
              COMPARE_OP              88 (bool(==))
              POP_JUMP_IF_FALSE        4 (to L3)
              NOT_TAKEN
              POP_TOP

  8           LOAD_CONST         

In [4]:
source = (
    "def label(status):\n"
    "    match status:\n"
    "        case _:\n"
    "            return 'go'\n"
    "        case 'red':\n"
    "            return 'stop'\n"
)
compile(source, "<match>", "exec")


SyntaxError: wildcard makes remaining patterns unreachable (<match>, line 3)

In [5]:
match True:
    case 1:
        print("matched")


matched


## When to use it

Use `match` when one subject is compared with several shapes, especially when two shapes share an arm. Use `if` and `elif` when the tests are different questions, such as `n < 0` or `name is None`. Use a dict when each value only needs to become another value and the arm has no extra steps. A dict will not capture a piece of the subject the way a pattern can.


## Common mistake

Writing a bare name when you meant the text in a variable. `case ok` does not compare with the string `"ok"`. It matches every status, stores it in `ok`, and the later code never gets a chance. `"missing"` prints as fine, and `ok` is no longer `"ok"`.


In [6]:
ok = "ok"
status = "missing"
match status:
    case ok:
        seen = "fine"
print(seen)
print(ok)


fine
missing


Quote the text you want to compare. A quoted pattern checks equality and does not store into `ok`. `_` then catches the rest without binding a name.


In [7]:
ok = "ok"
status = "missing"
match status:
    case "ok":
        seen = "fine"
    case _:
        seen = "other"
print(seen)
print(ok)


other
ok


## Try it

Change `light` so an unknown color comes back as that color, not as `"go"`. The leftover arm has to keep the value. Why is a bare name the right tool there, after it was the bug above?


## Recap

- `match` tries patterns from the top and stops at the first hit.
- `|` shares an arm. `_` is the leftover and binds nothing.
- A bare name matches everything and stores the subject.

**Next:** [practice](01-decisions-practice.ipynb)
